In [3]:
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

# Model name
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

# Load model + tokenizer
model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Pipeline (easy mode)
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

# Sample data
X_train = [
    "I've been waiting for a HuggingFace course my whole life.",
    "Python is great!"
]

# Pipeline prediction
res = classifier(X_train)
print("Pipeline output:")
print(res)

# Tokenization (manual step)
batch = tokenizer(
    X_train,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

print("\nTokenized input:")
print(batch)

# Model forward pass (manual mode)
with torch.no_grad():
    outputs = model(**batch) #unpack the batch because a dictionary
    print("\nRaw outputs:")
    print(outputs)

    predictions = F.softmax(outputs.logits, dim=1)
    print("\nProbabilities:")
    print(predictions)

    labels = torch.argmax(predictions, dim=1)
    print("\nPredicted labels:")
    print(labels)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Pipeline output:
[{'label': 'POSITIVE', 'score': 0.9598049521446228}, {'label': 'POSITIVE', 'score': 0.9998615980148315}]

Tokenized input:
{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101, 18750,  2003,  2307,   999,   102,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}

Raw outputs:
SequenceClassifierOutput(loss=None, logits=tensor([[-1.5607,  1.6123],
        [-4.2745,  4.6111]]), hidden_states=None, attentions=None)

Probabilities:
tensor([[4.0195e-02, 9.5980e-01],
        [1.3835e-04, 9.9986e-01]])

Predicted labels:
tensor([1, 1])
